# 03 - IA para la Detección de Malware

Este cuaderno implementa técnicas de ML para detectar malware analizando características estructurales de archivos ejecutables PE.

**Contenido:**
- Extracción de características de archivos PE
- Construcción del dataset de malware
- Clasificación con Árbol de Decisión
- Clasificación con Random Forest + SMOTE

## 4.1 Limitaciones del enfoque basado en firmas

Los sistemas antivirus tradicionales dependen de la detección basada en firmas, lo que los hace ineficaces frente a malware nuevo o polimórfico. La IA puede detectar malware analizando el **comportamiento** y las **características estructurales** de los archivos.

In [ ]:
# Instalar pefile si no está disponible
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pefile', '-q'])
print('[OK] pefile disponible')

## 4.2 Extracción de características de archivos PE

Los archivos ejecutables portátiles (PE) contienen metadatos ricos: número de secciones, punto de entrada, tablas de importaciones/exportaciones, entropía, etc.

In [ ]:
import pefile
import math
import os
import collections

def calcular_entropia(data: bytes) -> float:
    """Calcula la entropía de Shannon de un bloque de bytes.
    CORRECCIÓN: usa collections.Counter en lugar de bytes.count() O(n²)
    para rendimiento aceptable en archivos PE reales.
    """
    if not data:
        return 0.0
    n = len(data)
    counts = collections.Counter(data)
    return -sum((c / n) * math.log2(c / n) for c in counts.values() if c > 0)


def extraer_caracteristicas_pe(ruta_archivo: str) -> dict:
    """
    Extrae características relevantes de un ejecutable PE.
    Retorna un diccionario con las características o None si falla.
    """
    try:
        pe = pefile.PE(ruta_archivo)
    except pefile.PEFormatError:
        print(f'[!] No es un PE válido: {ruta_archivo}')
        return None

    caracteristicas = {}

    # Cabecera opcional
    caracteristicas['entry_point']        = pe.OPTIONAL_HEADER.AddressOfEntryPoint
    caracteristicas['image_base']         = pe.OPTIONAL_HEADER.ImageBase
    caracteristicas['size_of_image']      = pe.OPTIONAL_HEADER.SizeOfImage
    caracteristicas['size_code_section']  = pe.OPTIONAL_HEADER.SizeOfCode
    caracteristicas['dll_flag']           = pe.OPTIONAL_HEADER.DllCharacteristics

    # Secciones
    caracteristicas['num_sections'] = len(pe.sections)
    entropias = [calcular_entropia(sec.get_data()) for sec in pe.sections]
    caracteristicas['entropia_max']   = max(entropias) if entropias else 0.0
    caracteristicas['entropia_media'] = (
        sum(entropias) / len(entropias) if entropias else 0.0
    )

    # Importaciones / exportaciones
    if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
        caracteristicas['num_importaciones'] = sum(
            len(entry.imports) for entry in pe.DIRECTORY_ENTRY_IMPORT
        )
        caracteristicas['num_dlls_importadas'] = len(pe.DIRECTORY_ENTRY_IMPORT)
    else:
        caracteristicas['num_importaciones']   = 0
        caracteristicas['num_dlls_importadas'] = 0

    if hasattr(pe, 'DIRECTORY_ENTRY_EXPORT'):
        caracteristicas['num_exportaciones'] = len(
            pe.DIRECTORY_ENTRY_EXPORT.symbols
        )
    else:
        caracteristicas['num_exportaciones'] = 0

    # Tamaño del archivo
    caracteristicas['file_size'] = os.path.getsize(ruta_archivo)

    pe.close()
    return caracteristicas


# Uso de ejemplo (requiere un archivo .exe real)
ruta = 'muestra.exe'
if os.path.exists(ruta):
    feats = extraer_caracteristicas_pe(ruta)
    if feats:
        for k, v in feats.items():
            print(f'  {k:<28}: {v}')
else:
    print(f'Archivo "{ruta}" no encontrado. Proporciona un ejecutable PE real para extraer características.')

## 4.3 Construcción del dataset de malware

In [ ]:
import os
import pandas as pd
import numpy as np

def construir_dataset(directorios: dict) -> pd.DataFrame:
    """
    directorios: {'benign': '/ruta/benign', 'malicious': '/ruta/malicious'}
    Retorna un DataFrame con características y etiqueta.
    """
    registros = []
    for etiqueta, directorio in directorios.items():
        valor_label = 0 if etiqueta == 'benign' else 1
        if not os.path.isdir(directorio):
            print(f'[AVISO] Directorio no encontrado: {directorio}')
            continue
        for archivo in os.listdir(directorio):
            ruta_completa = os.path.join(directorio, archivo)
            feats = extraer_caracteristicas_pe(ruta_completa)
            if feats:
                feats['label'] = valor_label
                registros.append(feats)

    df = pd.DataFrame(registros)
    if not df.empty:
        print(f'Dataset: {len(df)} muestras | '
              f"Benign={df['label'].eq(0).sum()} | "
              f"Malicious={df['label'].eq(1).sum()}")
    return df


# ---------------------------------------------------------------
# Si no tienes directorios reales, se genera un dataset sintético
# ---------------------------------------------------------------
if not os.path.exists('file_features.csv'):
    print('Generando dataset sintético de características PE...')
    rng = np.random.default_rng(42)
    n_benign    = 800
    n_malicious = 200

    benign = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_benign),
        'image_base'         : rng.integers(0x400000, 0x500000, n_benign),
        'size_of_image'      : rng.integers(50000, 200000, n_benign),
        'size_code_section'  : rng.integers(10000, 80000, n_benign),
        'dll_flag'           : rng.integers(0, 256, n_benign),
        'num_sections'       : rng.integers(3, 7, n_benign),
        'entropia_max'       : rng.uniform(4.0, 6.5, n_benign),
        'entropia_media'     : rng.uniform(3.0, 5.5, n_benign),
        'num_importaciones'  : rng.integers(50, 200, n_benign),
        'num_dlls_importadas': rng.integers(3, 10, n_benign),
        'num_exportaciones'  : rng.integers(0, 20, n_benign),
        'file_size'          : rng.integers(50000, 500000, n_benign),
        'label'              : 0
    })
    malicious = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_malicious),
        'image_base'         : rng.integers(0x400000, 0x500000, n_malicious),
        'size_of_image'      : rng.integers(50000, 200000, n_malicious),
        'size_code_section'  : rng.integers(10000, 80000, n_malicious),
        'dll_flag'           : rng.integers(0, 256, n_malicious),
        'num_sections'       : rng.integers(5, 12, n_malicious),
        'entropia_max'       : rng.uniform(6.5, 8.0, n_malicious),
        'entropia_media'     : rng.uniform(5.5, 7.5, n_malicious),
        'num_importaciones'  : rng.integers(200, 500, n_malicious),
        'num_dlls_importadas': rng.integers(8, 20, n_malicious),
        'num_exportaciones'  : rng.integers(0, 5, n_malicious),
        'file_size'          : rng.integers(100000, 1000000, n_malicious),
        'label'              : 1
    })
    df_features = pd.concat([benign, malicious], ignore_index=True)
    df_features.to_csv('file_features.csv', index=False)
    print(f'Dataset sintético guardado: {len(df_features)} muestras.')
else:
    df_features = pd.read_csv('file_features.csv')
    print(f'Dataset cargado: {len(df_features)} muestras.')

print(df_features.head())

## 4.4.1 Árbol de Decisión

In [ ]:
# CORRECCIÓN: imports completos en la celda para que sea independiente
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Cargar datos
df = pd.read_csv('file_features.csv').dropna()
X = df.drop('label', axis=1)
y = df['label']

# 2. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Entrenar
clf = DecisionTreeClassifier(max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# 4. Evaluación
y_pred = clf.predict(X_test)
print('=== Reporte de clasificación ===')
print(classification_report(y_test, y_pred,
                             target_names=['Benign', 'Malicious']))

# Validación cruzada 5-fold
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='f1')
print(f'F1 CV (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# 5. Matriz de confusión
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malicious'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión — Árbol de Decisión')
plt.tight_layout()
plt.savefig('confusion_matrix_dt.png', dpi=150)
plt.show()

# 6. Importancia de características
importancias = pd.Series(clf.feature_importances_, index=X.columns)
top10 = importancias.nlargest(10)
top10.plot(kind='barh', color='steelblue', figsize=(8, 5))
plt.title('Top 10 características más importantes')
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Gráficos guardados: confusion_matrix_dt.png, feature_importance.png')

## 4.4.2 Random Forest con manejo de clases desbalanceadas (SMOTE)

In [ ]:
# CORRECCIÓN: imports completos en la celda para que sea independiente
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# 1. Cargar datos
df = pd.read_csv('file_features.csv').dropna()
X, y = df.drop('label', axis=1), df['label']

# 2. Balancear clases con SMOTE (Synthetic Minority Oversampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Antes SMOTE  : {y_train.value_counts().to_dict()}')
print(f'Después SMOTE: {y_train_res.value_counts().to_dict()}')

# 3. Entrenar Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_res, y_train_res)

# 4. Evaluación
y_pred_rf = rf.predict(X_test)
print('\n=== Reporte Random Forest + SMOTE ===')
print(classification_report(y_test, y_pred_rf,
                             target_names=['Benign', 'Malicious']))